#**1. Instalasi Library**

In [63]:
!pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#**2. Import Library**

In [64]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

#**3. Data Loading**

Memuat dataset dari GitHub

In [65]:
app_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/application_record.csv")
cred_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/credit_record.csv")

Membuat target fitur, yaitu is_bad sesuai dengan IRFS 9 basel iii 90 hari adalah nilai saat pinjaman harus segera di decline

In [ ]:
is_bad = (
    cred_df['STATUS']
    .isin(['3','4','5'])
    .groupby(cred_df['ID'])
    .max()  # Jika ada minimal 1 True, return 1
    .astype(int)
)

menambahkan fitur months balance min dan max untuk mempermudah analisis

In [ ]:
# 3. Hitung fitur temporal
credit_agg = cred_df.groupby('ID').agg(
    MONTHS_BALANCE_MIN=('MONTHS_BALANCE', 'min'),
    MONTHS_BALANCE_MAX=('MONTHS_BALANCE', 'max'),
    COUNT_LATE_LAST_12M=pd.NamedAgg(
        column='STATUS',
        aggfunc=lambda x: ((x.isin(['0','1','2','3','4','5'])) &
                          (cred_df.loc[x.index, 'MONTHS_BALANCE'] >= -12)).sum()
    )
).reset_index()

# Gabungkan target dan fitur temporal
credit_agg['TARGET'] = is_bad.values

# 4. Optimasi merge
merged_df = pd.merge(
    app_df,
    credit_agg,
    on='ID',
    how='inner'
)

# 5. Hitung fitur turunan
merged_df['CREDIT_HISTORY_LENGTH'] = (
    merged_df['MONTHS_BALANCE_MIN'] - merged_df['MONTHS_BALANCE_MAX']
)

In [68]:
credit_agg

,ID,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,COUNT_LATE_LAST_12M,TARGET
0,5001711,-3,0,3,0
1,5001712,-18,0,4,0
2,5001713,-21,0,0,0
3,5001714,-14,0,0,0
4,5001715,-59,0,0,0
...,...,...,...,...,...
45980,5150482,-28,-11,0,0
45981,5150483,-17,0,0,0
45982,5150484,-12,0,12,0
45983,5150485,-1,0,2,0


In [69]:
# 3. Merge Data
merged_df = app_df.merge(credit_agg, on='ID', how='inner')

# 4. Validasi Hasil Merge
print(f"Jumlah Data Awal (Application): {len(app_df)}")
print(f"Jumlah Data Setelah Merge: {len(merged_df)}")
print("\nDistribusi Target:")
print(merged_df['TARGET'].value_counts(normalize=True))

Jumlah Data Awal (Application): 438557
Jumlah Data Setelah Merge: 36457

Distribusi Target:
TARGET
0    0.991716
1    0.008284
Name: proportion, dtype: float64


In [70]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   36457 non-null  int64  
 1   CODE_GENDER          36457 non-null  object 
 2   FLAG_OWN_CAR         36457 non-null  object 
 3   FLAG_OWN_REALTY      36457 non-null  object 
 4   CNT_CHILDREN         36457 non-null  int64  
 5   AMT_INCOME_TOTAL     36457 non-null  float64
 6   NAME_INCOME_TYPE     36457 non-null  object 
 7   NAME_EDUCATION_TYPE  36457 non-null  object 
 8   NAME_FAMILY_STATUS   36457 non-null  object 
 9   NAME_HOUSING_TYPE    36457 non-null  object 
 10  DAYS_BIRTH           36457 non-null  int64  
 11  DAYS_EMPLOYED        36457 non-null  int64  
 12  FLAG_MOBIL           36457 non-null  int64  
 13  FLAG_WORK_PHONE      36457 non-null  int64  
 14  FLAG_PHONE           36457 non-null  int64  
 15  FLAG_EMAIL           36457 non-null 

In [71]:
merged_df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,...,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,COUNT_LATE_LAST_12M,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,1,1,0,0,NaN,2.0,-15,0,0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,1,1,0,0,NaN,2.0,-14,0,1,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,...,1,0,0,0,Security staff,2.0,-29,0,3,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,1,0,1,1,Sales staff,1.0,-4,0,2,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,1,0,1,1,Sales staff,1.0,-26,-22,0,0


In [72]:
for column in merged_df.select_dtypes(include='object'):
    unique_values = merged_df[column].unique()
    print(f"Unique values in {column}: {unique_values}")

Unique values in CODE_GENDER: ['M' 'F']
Unique values in FLAG_OWN_CAR: ['Y' 'N']
Unique values in FLAG_OWN_REALTY: ['Y' 'N']
Unique values in NAME_INCOME_TYPE: ['Working' 'Commercial associate' 'Pensioner' 'State servant' 'Student']
Unique values in NAME_EDUCATION_TYPE: ['Higher education' 'Secondary / secondary special' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Unique values in NAME_FAMILY_STATUS: ['Civil marriage' 'Married' 'Single / not married' 'Separated' 'Widow']
Unique values in NAME_HOUSING_TYPE: ['Rented apartment' 'House / apartment' 'Municipal apartment'
 'With parents' 'Co-op apartment' 'Office apartment']
Unique values in OCCUPATION_TYPE: [nan 'Security staff' 'Sales staff' 'Accountants' 'Laborers' 'Managers'
 'Drivers' 'Core staff' 'High skill tech staff' 'Cleaning staff'
 'Private service staff' 'Cooking staff' 'Low-skill Laborers'
 'Medicine staff' 'Secretaries' 'Waiters/barmen staff' 'HR staff'
 'Realty agents' 'IT staff']


melakukan imputasi missing value di occupation menjadi object stirnf 'Unknown'

In [73]:
# Cek persentase missing values
missing_report = merged_df.isnull().mean() * 100
print("Missing Values Report:")
print(missing_report[missing_report > 0])

# Handle OCCUPATION_TYPE (kolom dengan missing tertinggi)
merged_df['OCCUPATION_TYPE'] = merged_df['OCCUPATION_TYPE'].fillna('Unknown')

Missing Values Report:
OCCUPATION_TYPE    31.058507
dtype: float64


menambahkan fitur age dan years employed yang mungkin lebih berguna saat visualisasi nanti

In [75]:
# Usia dalam tahun (DAYS_BIRTH bernilai negatif)
merged_df['AGE'] = (-merged_df['DAYS_BIRTH'] / 365).round(1)

# Lama bekerja (DAYS_EMPLOYED)
merged_df['YEARS_EMPLOYED'] = (
    merged_df['DAYS_EMPLOYED']
    .apply(lambda x: -x/365 if x < 0 else 0)
    .round(1)
)

melakukan binning amt income total dan age dengan manambahkan fitur income group dan age group mempermudah visualisaias

In [76]:
# Binning pendapatan
merged_df['INCOME_GROUP'] = pd.cut(
    merged_df['AMT_INCOME_TOTAL'],
    bins=[0, 30000, 60000, 100000, np.inf],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Binning usia
merged_df['AGE_GROUP'] = pd.cut(
    merged_df['AGE'],
    bins=[18, 30, 40, 50, 60, 100],
    labels=['18-29', '30-39', '40-49', '50-59', '60+']
)

mengubah flag yang sebelumnya object menjadi binary

In [ ]:
# Binary encoding untuk flag
binary_mapping = {'Y': 1, 'N': 0}
merged_df['FLAG_OWN_CAR'] = merged_df['FLAG_OWN_CAR'].map(binary_mapping)
merged_df['FLAG_OWN_REALTY'] = merged_df['FLAG_OWN_REALTY'].map(binary_mapping)



membuat label encoder untuk education level

In [ ]:
# Label encoding untuk pendidikan
education_order = [
    'Lower secondary', 
    'Secondary / secondary special', 
    'Incomplete higher', 
    'Higher education', 
    'Academic degree'
]
merged_df['EDUCATION_RANK'] = merged_df['NAME_EDUCATION_TYPE'].map(
    {k:v for v,k in enumerate(education_order)}
)

membersihkan outlier

In [78]:
# Deteksi outlier AMT_INCOME_TOTAL
Q1 = merged_df['AMT_INCOME_TOTAL'].quantile(0.25)
Q3 = merged_df['AMT_INCOME_TOTAL'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

# Cap outliers
merged_df['AMT_INCOME_TOTAL_CAPPED'] = merged_df['AMT_INCOME_TOTAL'].clip(
    lower=lower_bound, 
    upper=upper_bound
)

# Atau hapus outliers
# merged_df = merged_df.query('AMT_INCOME_TOTAL >= @lower_bound & AMT_INCOME_TOTAL <= @upper_bound')

In [80]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   ID                       36457 non-null  int64   
 1   CODE_GENDER              36457 non-null  object  
 2   FLAG_OWN_CAR             36457 non-null  int64   
 3   FLAG_OWN_REALTY          36457 non-null  int64   
 4   CNT_CHILDREN             36457 non-null  int64   
 5   AMT_INCOME_TOTAL         36457 non-null  float64 
 6   NAME_INCOME_TYPE         36457 non-null  object  
 7   NAME_EDUCATION_TYPE      36457 non-null  object  
 8   NAME_FAMILY_STATUS       36457 non-null  object  
 9   NAME_HOUSING_TYPE        36457 non-null  object  
 10  DAYS_BIRTH               36457 non-null  int64   
 11  DAYS_EMPLOYED            36457 non-null  int64   
 12  FLAG_MOBIL               36457 non-null  int64   
 13  FLAG_WORK_PHONE          36457 non-null  int64   
 14  FLAG_P

isi merged dataset yang akan digunakan untuk eda

In [82]:
merged_df.columns.tolist()

['ID',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'FLAG_MOBIL',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'MONTHS_BALANCE_MIN',
 'MONTHS_BALANCE_MAX',
 'COUNT_LATE_LAST_12M',
 'TARGET',
 'AGE',
 'YEARS_EMPLOYED',
 'INCOME_GROUP',
 'AGE_GROUP',
 'EDUCATION_RANK',
 'AMT_INCOME_TOTAL_CAPPED',
 'CREDIT_HISTORY_LENGTH']

save merged dataset ke lokal

In [ ]:
# merged_df.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\merged_dataset.csv", index=False)